In [2]:
from bigmodule import M, I
import dai
import pandas as pd
import numpy as np
import os
from datetime import timedelta


# ============================================================
# 日期
# ============================================================

start_date_str = "2020-01-01"
end_date_str = "2026-04-01"

DATA_START = "2019-01-01"

# 只做主板
LIST_SECTOR_MAINBOARD = 1

# 每个调仓日只取主板小市值前 N 只
# 如果仍然内存压力大，先改成 400
CANDIDATE_SIZE_RANK = 600

# 调仓频率，用于构造研究面板
PANEL_REBALANCE_DAYS = 20

# 需要的历史/未来窗口
LOOKBACK_DAYS = 220
FORWARD_DAYS = 60

# 基础过滤
MIN_LIST_DAYS = 365
MIN_TRADING_DAYS = 240

# 缓存
PANEL_CACHE_PATH = "/home/aiuser/work/mainboard_smallcap_panel.pkl"
USE_PANEL_CACHE = True

BENCHMARK = "上证指数"

In [3]:
def load_trade_dates():
    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_factors_base
    WHERE date >= '{DATA_START}'
      AND date <= '{end_date_str}'
    ORDER BY date
    """

    trade_dates = dai.query(sql).df()
    trade_dates["date"] = pd.to_datetime(trade_dates["date"]).dt.strftime("%Y-%m-%d")
    trade_dates = trade_dates.drop_duplicates().sort_values("date").reset_index(drop=True)

    return trade_dates


def make_signal_dates(trade_dates, rebalance_days=PANEL_REBALANCE_DAYS):
    x = trade_dates[trade_dates["date"] >= start_date_str].copy().reset_index(drop=True)
    x["idx"] = np.arange(len(x))

    signal_dates = list(
        x.loc[x["idx"] % rebalance_days == 0, "date"]
    )

    return signal_dates


trade_dates = load_trade_dates()
signal_dates = make_signal_dates(trade_dates, PANEL_REBALANCE_DAYS)

print("trade_dates:", trade_dates.shape)
print("signal_dates:", len(signal_dates))
print(signal_dates[:10])

trade_dates: (1756, 1)
signal_dates: 76
['2020-01-02', '2020-02-07', '2020-03-06', '2020-04-03', '2020-05-07', '2020-06-04', '2020-07-06', '2020-08-03', '2020-08-31', '2020-09-28']


In [4]:
def get_window_dates(signal_date, trade_dates, lookback_days=LOOKBACK_DAYS, forward_days=FORWARD_DAYS):
    dates = list(trade_dates["date"])
    idx = dates.index(signal_date)

    start_idx = max(0, idx - lookback_days)
    end_idx = min(len(dates) - 1, idx + forward_days)

    return dates[start_idx], dates[end_idx]


def load_one_signal_panel(signal_date):
    """
    对单个调仓信号日：
    1. 找主板小市值候选股票；
    2. 拉过去 LOOKBACK_DAYS 到未来 FORWARD_DAYS 的价格窗口；
    3. 计算因子；
    4. 只保留 signal_date 当天的截面。
    """

    # --------------------------------------------------------
    # 1. 候选股票：只取 signal_date 当天主板小市值前 N
    # --------------------------------------------------------
    candidate_sql = f"""
    SELECT instrument
    FROM cn_stock_factors_base
    WHERE date = '{signal_date}'
      AND list_sector = {LIST_SECTOR_MAINBOARD}
      AND st_status = 0
      AND suspended = 0
      AND list_days > {MIN_LIST_DAYS}
      AND trading_days > {MIN_TRADING_DAYS}
      AND float_market_cap > 0
    ORDER BY float_market_cap ASC
    LIMIT {CANDIDATE_SIZE_RANK}
    """

    candidate_df = dai.query(candidate_sql).df()
    candidates = candidate_df["instrument"].dropna().astype(str).unique().tolist()

    if len(candidates) == 0:
        return pd.DataFrame()

    instruments_sql = ",".join([f"'{x}'" for x in candidates])

    # --------------------------------------------------------
    # 2. 拉窗口数据
    # --------------------------------------------------------
    window_start, window_end = get_window_dates(signal_date, trade_dates)

    raw_sql = f"""
    SELECT
        b.date,
        b.instrument,

        b.close,
        b.amount,
        b.volume,
        b.open,
        b.high,
        b.low,

        b.price_limit_status,
        b.st_status,
        b.suspended,
        b.list_days,
        b.trading_days,

        b.list_sector,
        b.total_market_cap,
        b.float_market_cap,
        b.sw2021_level1,

        v.pb,
        v.pe_ttm,
        v.pcf_op_ttm,
        v.pcf_net_ttm,

        f.operating_net_income_ttm,
        f.net_profit_deducted_lf,
        f.fcff_ttm,
        f.fcfe_ttm,
        f.ebit_ttm,
        f.ebitda_ttm,
        f.nopat_ttm,
        f.invested_capital_lf

    FROM cn_stock_factors_base AS b

    LEFT JOIN cn_stock_valuation AS v
        ON b.date = v.date
       AND b.instrument = v.instrument

    LEFT JOIN cn_stock_factors_financial_indicators AS f
        ON b.date = f.date
       AND b.instrument = f.instrument

    WHERE b.date >= '{window_start}'
      AND b.date <= '{window_end}'
      AND b.instrument IN ({instruments_sql})
      AND b.list_sector = {LIST_SECTOR_MAINBOARD}
    """

    df = dai.query(raw_sql).df()

    if df.empty:
        return pd.DataFrame()

    df = df.dropna(subset=["date", "instrument", "close"]).copy()
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["instrument"] = df["instrument"].astype(str)

    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    # 降低内存
    float_cols = [
        "close",
        "amount",
        "volume",
        "open",
        "high",
        "low",
        "total_market_cap",
        "float_market_cap",
        "pb",
        "pe_ttm",
        "pcf_op_ttm",
        "pcf_net_ttm",
        "operating_net_income_ttm",
        "net_profit_deducted_lf",
        "fcff_ttm",
        "fcfe_ttm",
        "ebit_ttm",
        "ebitda_ttm",
        "nopat_ttm",
        "invested_capital_lf",
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    int_cols = [
        "price_limit_status",
        "st_status",
        "suspended",
        "list_days",
        "trading_days",
        "list_sector",
    ]

    for col in int_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --------------------------------------------------------
    # 3. 计算滚动因子
    # --------------------------------------------------------
    g = df.groupby("instrument")

    df["ret_120d"] = g["close"].transform(lambda x: x / x.shift(120) - 1)
    df["ret_60d"] = g["close"].transform(lambda x: x / x.shift(60) - 1)
    df["ret_20d"] = g["close"].transform(lambda x: x / x.shift(20) - 1)
    df["ret_5d"] = g["close"].transform(lambda x: x / x.shift(5) - 1)

    df["daily_ret"] = g["close"].transform(lambda x: x / x.shift(1) - 1)
    df["amount_to_float_mv"] = df["amount_20d"] / df["float_market_cap"]
    df["vol_20d"] = g["daily_ret"].transform(
        lambda x: x.rolling(20, min_periods=15).std()
    )

    df["vol_60d"] = g["daily_ret"].transform(
        lambda x: x.rolling(60, min_periods=40).std()
    )

    df["amount_20d"] = g["amount"].transform(
        lambda x: x.rolling(20, min_periods=10).mean()
    )

    df["ma_20"] = g["close"].transform(
        lambda x: x.rolling(20, min_periods=15).mean()
    )

    df["ma_60"] = g["close"].transform(
        lambda x: x.rolling(60, min_periods=40).mean()
    )

    df["operating_yield"] = df["operating_net_income_ttm"] / df["total_market_cap"]

    # 未来收益，用于因子测试
    df["fwd_20d"] = g["close"].transform(lambda x: x.shift(-20) / x - 1)
    df["fwd_30d"] = g["close"].transform(lambda x: x.shift(-30) / x - 1)
    df["fwd_40d"] = g["close"].transform(lambda x: x.shift(-40) / x - 1)

    # --------------------------------------------------------
    # 4. 只保留 signal_date 当天截面
    # --------------------------------------------------------
    panel = df[df["date"] == signal_date].copy()

    # 防御：只保留当天候选股票
    panel = panel[panel["instrument"].isin(candidates)].copy()

    panel["signal_date"] = signal_date

    return panel

In [5]:
def build_panel(use_cache=True):
    if use_cache and os.path.exists(PANEL_CACHE_PATH):
        panel_df = pd.read_pickle(PANEL_CACHE_PATH)
        print("loaded panel cache:", PANEL_CACHE_PATH)
        print("panel_df shape:", panel_df.shape)
        return panel_df

    panels = []

    for i, d in enumerate(signal_dates):
        try:
            one = load_one_signal_panel(d)

            if len(one) > 0:
                panels.append(one)

            print(f"[{i + 1}/{len(signal_dates)}] {d}, shape={one.shape}")

        except Exception as e:
            print(f"[ERROR] {d}: {e}")
            continue

    if len(panels) == 0:
        raise ValueError("No panel data generated.")

    panel_df = pd.concat(panels, axis=0, ignore_index=True)

    panel_df = panel_df.sort_values(["date", "instrument"]).reset_index(drop=True)

    panel_df.to_pickle(PANEL_CACHE_PATH)

    print("saved panel cache:", PANEL_CACHE_PATH)
    print("panel_df shape:", panel_df.shape)

    return panel_df


panel_df = build_panel(USE_PANEL_CACHE)

[1/76] 2020-01-02, shape=(600, 44)
[2/76] 2020-02-07, shape=(600, 44)
[3/76] 2020-03-06, shape=(600, 44)
[4/76] 2020-04-03, shape=(600, 44)
[5/76] 2020-05-07, shape=(600, 44)
[6/76] 2020-06-04, shape=(600, 44)
[7/76] 2020-07-06, shape=(600, 44)
[8/76] 2020-08-03, shape=(600, 44)
[9/76] 2020-08-31, shape=(600, 44)
[10/76] 2020-09-28, shape=(600, 44)
[11/76] 2020-11-03, shape=(600, 44)
[12/76] 2020-12-01, shape=(600, 44)
[13/76] 2020-12-29, shape=(600, 44)
[14/76] 2021-01-27, shape=(600, 44)
[15/76] 2021-03-03, shape=(600, 44)
[16/76] 2021-03-31, shape=(600, 44)
[17/76] 2021-04-29, shape=(600, 44)
[18/76] 2021-06-01, shape=(600, 44)
[19/76] 2021-06-30, shape=(600, 44)
[20/76] 2021-07-28, shape=(600, 44)
[21/76] 2021-08-25, shape=(600, 44)
[22/76] 2021-09-24, shape=(600, 44)
[23/76] 2021-10-29, shape=(600, 44)
[24/76] 2021-11-26, shape=(600, 44)
[25/76] 2021-12-24, shape=(600, 44)
[26/76] 2022-01-24, shape=(600, 44)
[27/76] 2022-02-28, shape=(600, 44)
[28/76] 2022-03-28, shape=(600, 44)
[

In [6]:
def apply_base_filter(
    df,
    require_pcf_positive=False,
    require_profit=False,
    use_trend_filter=False,
    use_vol_filter=False,
    use_reversal_band=False,
    min_liquidity_pct=0.20,
):
    x = df.copy()

    x = x[
        (x["list_sector"] == LIST_SECTOR_MAINBOARD) &
        (x["list_days"] > MIN_LIST_DAYS) &
        (x["trading_days"] > MIN_TRADING_DAYS) &
        (x["st_status"] == 0) &
        (x["suspended"] == 0) &
        (x["price_limit_status"] == 2) &
        (x["float_market_cap"] > 0) &
        (x["total_market_cap"] > 0) &
        (x["amount_20d"] > 0) &
        (x["ret_60d"].notna()) &
        (x["ret_20d"].notna()) &
        (x["ret_5d"].notna()) &
        (x["vol_20d"].notna()) &
        (x["ma_20"].notna()) &
        (x["ma_60"].notna()) &
        (x["sw2021_level1"].notna())
    ].copy()

    # PB / PE 默认只要求存在，不强制正数
    x = x[
        (x["pb"].notna()) &
        (x["pe_ttm"].notna())
    ].copy()

    if require_pcf_positive:
        x = x[(x["pcf_op_ttm"] > 0)].copy()

    if require_profit:
        x = x[
            (
                (x["operating_net_income_ttm"].fillna(0) > 0) |
                (x["net_profit_deducted_lf"].fillna(0) > 0) |
                (x["fcff_ttm"].fillna(0) > 0)
            )
        ].copy()

    x["liquidity_rank"] = x.groupby("date")["amount_20d"].rank(
        pct=True,
        ascending=True,
    )

    x = x[x["liquidity_rank"] >= min_liquidity_pct].copy()

    if use_reversal_band:
        x["rev_pct"] = x.groupby("date")["ret_60d"].rank(
            pct=True,
            ascending=True,
        )
        x = x[(x["rev_pct"] >= 0.10) & (x["rev_pct"] <= 0.80)].copy()

    if use_trend_filter:
        x["trend_ok"] = (
            (x["close"] > x["ma_20"] * 0.92) |
            (x["close"] > x["ma_60"] * 0.92)
        )
        x = x[x["trend_ok"]].copy()

    if use_vol_filter:
        x["vol_pct"] = x.groupby("date")["vol_20d"].rank(
            pct=True,
            ascending=True,
        )
        x = x[x["vol_pct"] <= 0.90].copy()

    return x


def add_factor_ranks(df):
    x = df.copy()

    x["rank_size"] = x.groupby("date")["float_market_cap"].rank(
        pct=True,
        ascending=True,
    )

    x["rank_lowvol"] = x.groupby("date")["vol_20d"].rank(
        pct=True,
        ascending=True,
    )

    x["rank_pb"] = x.groupby("date")["pb"].rank(
        pct=True,
        ascending=True,
    )

    x["rank_pcf"] = x.groupby("date")["pcf_op_ttm"].rank(
        pct=True,
        ascending=True,
    )

    x["rank_rev60"] = x.groupby("date")["ret_60d"].rank(
        pct=True,
        ascending=True,
    )

    # 60日反转分位：ret_60d 越低越弱
    x["rev60_pct"] = x.groupby("date")["ret_60d"].rank(
        pct=True,
        ascending=True,
    )

    # 中度反转：目标区间大约 20%~60%，偏离 40% 越近越好
    x["mid_rev60_score_raw"] = (x["rev60_pct"] - 0.40).abs()

    x["rank_mid_rev60"] = x.groupby("date")["mid_rev60_score_raw"].rank(
        pct=True,
        ascending=True,
    )

    x["rank_mom20"] = x.groupby("date")["ret_20d"].rank(
        pct=True,
        ascending=False,
    )

    x["rank_mom60"] = x.groupby("date")["ret_60d"].rank(
        pct=True,
        ascending=False,
    )

    x["rank_quality"] = x.groupby("date")["operating_yield"].rank(
        pct=True,
        ascending=False,
    )
    
    x["rank_low_turnover_proxy"] = x.groupby("date")["amount_to_float_mv"].rank(
        pct=True,
        ascending=True,
    )

    return x


research_base = apply_base_filter(
    panel_df,
    require_pcf_positive=False,
    require_profit=False,
    use_trend_filter=False,
    use_vol_filter=False,
    use_reversal_band=False,
    min_liquidity_pct=0.20,
)

research_ranked = add_factor_ranks(research_base)

print("research_ranked shape:", research_ranked.shape)

research_ranked shape: (35663, 53)


In [7]:
def factor_quantile_report(
    df,
    factor_col,
    forward_col="fwd_20d",
    n_quantiles=5,
    ascending=True,
    min_count_per_date=50,
):
    cols = ["date", "instrument", factor_col, forward_col]
    x = df[cols].dropna().copy()

    date_counts = x.groupby("date")["instrument"].count()
    valid_dates = date_counts[date_counts >= min_count_per_date].index
    x = x[x["date"].isin(valid_dates)].copy()

    def assign_quantile(s):
        rank = s.rank(method="first", ascending=ascending)
        try:
            return pd.qcut(rank, n_quantiles, labels=False) + 1
        except ValueError:
            return pd.Series(np.nan, index=s.index)

    x["q"] = x.groupby("date")[factor_col].transform(assign_quantile)
    x = x.dropna(subset=["q"]).copy()
    x["q"] = x["q"].astype(int)

    report = (
        x.groupby("q")[forward_col]
        .agg(["mean", "median", "std", "count"])
        .reset_index()
    )

    if 1 in set(report["q"]) and n_quantiles in set(report["q"]):
        q1 = report.loc[report["q"] == 1, "mean"].iloc[0]
        qn = report.loc[report["q"] == n_quantiles, "mean"].iloc[0]
        print(f"{factor_col}, {forward_col}, Q1-Q{n_quantiles}:", q1 - qn)

    return report

In [8]:
# 纯小市值：rank_size 越小越好
factor_quantile_report(
    research_ranked,
    factor_col="rank_size",
    forward_col="fwd_20d",
    ascending=True,
)

rank_size, fwd_20d, Q1-Q5: 0.0026286673


,q,mean,median,std,count
0,1,0.020706,0.011105,0.148235,7063
1,2,0.021175,0.012440,0.153675,7020
2,3,0.021528,0.010324,0.148780,7017
3,4,0.017028,0.007908,0.146690,7020
4,5,0.018078,0.006022,0.153761,7051


In [9]:
# 低波动：rank_lowvol 越小越好
factor_quantile_report(
    research_ranked,
    factor_col="rank_lowvol",
    forward_col="fwd_20d",
    ascending=True,
)

rank_lowvol, fwd_20d, Q1-Q5: 0.011304399


,q,mean,median,std,count
0,1,0.021010,0.017028,0.126185,7063
1,2,0.022084,0.015406,0.132480,7020
2,3,0.024392,0.012674,0.156736,7017
3,4,0.021360,0.006065,0.157381,7020
4,5,0.009705,-0.009594,0.173120,7051


In [10]:
# 低 PB
factor_quantile_report(
    research_ranked,
    factor_col="rank_pb",
    forward_col="fwd_20d",
    ascending=True,
)

rank_pb, fwd_20d, Q1-Q5: 0.010460156


,q,mean,median,std,count
0,1,0.023632,0.016596,0.141789,7063
1,2,0.022079,0.012389,0.144441,7020
2,3,0.019148,0.009980,0.147315,7017
3,4,0.020488,0.005625,0.154600,7020
4,5,0.013172,0.001114,0.162068,7051


In [11]:
# 60日反转
factor_quantile_report(
    research_ranked,
    factor_col="rank_rev60",
    forward_col="fwd_20d",
    ascending=True,
)

rank_rev60, fwd_20d, Q1-Q5: 0.01147161


,q,mean,median,std,count
0,1,0.021066,0.013055,0.152547,7063
1,2,0.023597,0.017103,0.136913,7020
2,3,0.024535,0.013782,0.141870,7017
3,4,0.019759,0.007978,0.148477,7020
4,5,0.009594,-0.008902,0.168937,7051


In [12]:
# 20日动量
factor_quantile_report(
    research_ranked,
    factor_col="rank_mom20",
    forward_col="fwd_20d",
    ascending=True,
)

rank_mom20, fwd_20d, Q1-Q5: -0.0140359625


,q,mean,median,std,count
0,1,0.006989,-0.009662,0.169458,7063
1,2,0.021289,0.008566,0.145416,7020
2,3,0.022686,0.013830,0.142801,7017
3,4,0.026597,0.019393,0.142897,7020
4,5,0.021025,0.012780,0.148224,7051


In [ ]:
STRATEGIES = {
    # 1. 纯小市值：最重要的对照组
    "pure_size": {
        "weights": {
            "rank_size": 1.00,
        },
        "require_pcf_positive": False,
        "require_profit": False,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 2. 小市值 + 低波动
    "size_lowvol": {
        "weights": {
            "rank_size": 0.70,
            "rank_lowvol": 0.30,
        },
        "require_pcf_positive": False,
        "require_profit": False,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 3. 小市值 + 动量
    "size_mom20": {
        "weights": {
            "rank_size": 0.70,
            "rank_mom20": 0.30,
        },
        "require_pcf_positive": False,
        "require_profit": False,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 4. 小市值 + 反转
    "size_reversal": {
        "weights": {
            "rank_size": 0.70,
            "rank_rev60": 0.30,
        },
        "require_pcf_positive": False,
        "require_profit": False,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": True,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 5. 小市值 + 估值
    "size_value": {
        "weights": {
            "rank_size": 0.60,
            "rank_pb": 0.20,
            "rank_pcf": 0.20,
        },
        "require_pcf_positive": True,
        "require_profit": False,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 6. 小市值 + 质量
    "size_quality": {
        "weights": {
            "rank_size": 0.70,
            "rank_quality": 0.30,
        },
        "require_pcf_positive": False,
        "require_profit": True,
        "use_trend_filter": False,
        "use_vol_filter": False,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },

    # 7. 相对均衡版
    "balanced": {
        "weights": {
            "rank_size": 0.50,
            "rank_lowvol": 0.15,
            "rank_mom20": 0.15,
            "rank_pb": 0.10,
            "rank_quality": 0.10,
        },
        "require_pcf_positive": False,
        "require_profit": False,
        "use_trend_filter": True,
        "use_vol_filter": True,
        "use_reversal_band": False,
        "hold_num": 5,
        "buffer_num": 15,
        "rebalance_days": 20,
        "exposure": 1.00,
    },
}

def get_trade_dates(df):
    trade_dates = (
        df[["date"]]
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
    )
    trade_dates["next_date"] = trade_dates["date"].shift(-1)
    return trade_dates


def make_rebalance_dates(trade_dates, rebalance_days):
    trade_date_df = (
        trade_dates[trade_dates["date"] >= start_date_str]
        .copy()
        .reset_index(drop=True)
    )

    trade_date_df["rebalance_index"] = np.arange(len(trade_date_df))

    rebalance_dates = set(
        trade_date_df.loc[
            trade_date_df["rebalance_index"] % rebalance_days == 0,
            "date"
        ]
    )

    return rebalance_dates


def select_with_industry_limit(df_day, last_holdings, hold_num, buffer_num):
    df_day = df_day.sort_values("score").copy()
    df_day["score_rank"] = np.arange(1, len(df_day) + 1)

    selected_rows = []
    selected_set = set()
    industry_count = {}

    # 先保留旧持仓
    keep_candidates = df_day[
        (df_day["instrument"].isin(last_holdings)) &
        (df_day["score_rank"] <= buffer_num)
    ].copy()

    for _, row in keep_candidates.iterrows():
        inst = row["instrument"]
        ind = row["sw2021_level1"]

        if industry_count.get(ind, 0) >= MAX_INDUSTRY_COUNT:
            continue

        selected_rows.append(row)
        selected_set.add(inst)
        industry_count[ind] = industry_count.get(ind, 0) + 1

        if len(selected_rows) >= hold_num:
            break

    # 再补新股
    for _, row in df_day.iterrows():
        if len(selected_rows) >= hold_num:
            break

        inst = row["instrument"]
        ind = row["sw2021_level1"]

        if inst in selected_set:
            continue

        if industry_count.get(ind, 0) >= MAX_INDUSTRY_COUNT:
            continue

        selected_rows.append(row)
        selected_set.add(inst)
        industry_count[ind] = industry_count.get(ind, 0) + 1

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=df_day.columns)

    return pd.DataFrame(selected_rows)


def build_selected(feature_df, strategy_name, strategy):
    print("=" * 80)
    print("build strategy:", strategy_name)
    print(strategy)

    df = apply_base_filter(
        feature_df,
        require_pcf_positive=strategy.get("require_pcf_positive", False),
        require_profit=strategy.get("require_profit", False),
        use_trend_filter=strategy.get("use_trend_filter", False),
        use_vol_filter=strategy.get("use_vol_filter", False),
        use_reversal_band=strategy.get("use_reversal_band", False),
        min_liquidity_pct=strategy.get("min_liquidity_pct", MIN_LIQUIDITY_PCT),
    )

    df = add_factor_ranks(df)

    # 生成综合 score
    df["score"] = 0.0

    for rank_col, weight in strategy["weights"].items():
        if rank_col not in df.columns:
            raise ValueError(f"rank column not found: {rank_col}")
        df["score"] += weight * df[rank_col]

    df = df.dropna(subset=["score"]).copy()

    print("after filter and score:", df.shape)

    trade_dates = get_trade_dates(feature_df)
    date_map = dict(zip(trade_dates["date"], trade_dates["next_date"]))

    df["signal_date"] = df["date"]
    df["trade_date"] = df["signal_date"].map(date_map)
    df = df.dropna(subset=["trade_date"]).copy()
    df = df[df["trade_date"] >= start_date_str].copy()

    rebalance_dates = make_rebalance_dates(
        trade_dates,
        strategy.get("rebalance_days", DEFAULT_REBALANCE_DAYS),
    )

    df = df[df["trade_date"].isin(rebalance_dates)].copy()
    df = df.sort_values(["trade_date", "score"]).copy()

    print("after rebalance filter:", df.shape)

    hold_num = strategy.get("hold_num", DEFAULT_HOLD_NUM)
    buffer_num = strategy.get("buffer_num", DEFAULT_BUFFER_NUM)
    exposure = strategy.get("exposure", 1.0)

    selected_rows = []
    last_holdings = set()

    for trade_date, df_day in df.groupby("trade_date", sort=True):
        final = select_with_industry_limit(
            df_day=df_day,
            last_holdings=last_holdings,
            hold_num=hold_num,
            buffer_num=buffer_num,
        )

        if len(final) == 0:
            continue

        final = final.head(hold_num).copy()
        final["date"] = trade_date
        final["position"] = exposure / len(final)
        final["strategy"] = strategy_name

        selected_rows.append(
            final[["date", "instrument", "position", "score", "strategy"]]
        )

        last_holdings = set(final["instrument"])

    if len(selected_rows) == 0:
        raise ValueError(f"No selected stocks for strategy: {strategy_name}")

    selected = pd.concat(selected_rows, axis=0)

    selected["date"] = pd.to_datetime(selected["date"])
    selected["instrument"] = selected["instrument"].astype(str)
    selected["position"] = selected["position"].astype(float)
    selected["score"] = selected["score"].astype(float)

    selected = selected[["date", "instrument", "position", "score", "strategy"]]
    selected = selected.sort_values(["date", "score"]).reset_index(drop=True)

    print("selected shape:", selected.shape)
    print("date range:", selected["date"].min(), selected["date"].max())
    print(selected.head(20))

    return selected

selected_pure_size = build_selected(
    feature_df,
    "pure_size",
    STRATEGIES["pure_size"],
)

In [ ]:
selected_pure_size = build_selected(
    feature_df,
    "pure_size",
    STRATEGIES["pure_size"],
)

In [ ]:
selected_size_mom20 = build_selected(
    feature_df,
    "size_mom20",
    STRATEGIES["size_mom20"],
)

In [ ]:
def inspect_selected(selected):
    x = selected.copy()
    x["date"] = pd.to_datetime(x["date"])

    print("shape:", x.shape)
    print("date range:", x["date"].min(), x["date"].max())

    print("\npositions per date:")
    print(x.groupby("date")["instrument"].count().describe())

    print("\nposition sum per date:")
    print(x.groupby("date")["position"].sum().describe())

    print("\nfirst dates:")
    print(x.head(30))

    return x


inspect_selected(selected_pure_size)

In [ ]:
def run_backtest(selected, strategy_name, plot_charts=True):
    selected_to_write = selected[["date", "instrument", "position", "score"]].copy()

    # data date 必须是 datetime
    selected_to_write["date"] = pd.to_datetime(selected_to_write["date"])
    selected_to_write["instrument"] = selected_to_write["instrument"].astype(str)
    selected_to_write["position"] = selected_to_write["position"].astype(float)
    selected_to_write["score"] = selected_to_write["score"].astype(float)

    stock_data = dai.DataSource.write_bdb(selected_to_write)

    m5 = M.bigtrader.v30(
        data=stock_data,
        start_date=start_date_str,
        end_date=end_date_str,
        initialize=m5_initialize_bigquant_run,
        before_trading_start=m5_before_trading_start_bigquant_run,
        handle_tick=m5_handle_tick_bigquant_run,
        handle_data=m5_handle_data_bigquant_run,
        handle_trade=m5_handle_trade_bigquant_run,
        handle_order=m5_handle_order_bigquant_run,
        after_trading=m5_after_trading_bigquant_run,
        capital_base=500000,
        frequency="daily",
        product_type="股票",

        rebalance_period_type="交易日",
        rebalance_period_days="1",
        rebalance_period_roll_forward=True,

        backtest_engine_mode="标准模式",
        before_start_days=0,
        volume_limit=1,
        order_price_field_buy="open",
        order_price_field_sell="open",
        benchmark=BENCHMARK,
        plot_charts=plot_charts,
        debug=False,
        backtest_only=False,
        m_name=f"主板_因子测试_{strategy_name}",
    )

    return m5

In [ ]:
m_pure_size = run_backtest(
    selected_pure_size,
    "pure_size",
    plot_charts=True,
)

In [ ]:
m_size_lowvol = run_backtest(
    selected_size_lowvol,
    "size_lowvol",
    plot_charts=True,
)

In [ ]:
m_size_mom20 = run_backtest(
    selected_size_mom20,
    "size_mom20",
    plot_charts=True,
)